In [ ]:
import pandas as pd, numpy as np

df = pd.read_excel('/content/oasis_cross-sectional-5708aa0a98d82080.xlsx')

def to_label(cdr):
    if pd.isna(cdr) or cdr == 0: return 0
    if cdr == 0.5:               return 1
    return 2

# cohort A: the original buggy mapping (blank CDR -> CN)
A = df.copy()
A['label'] = A['CDR'].apply(to_label)

# cohort B: corrected (CDR-rated subjects only)
B = df[df['CDR'].notna()].copy()
B['label'] = B['CDR'].apply(to_label)

for name, c in [('A buggy', A), ('B corrected', B)]:
    print(f'{name}: n={len(c)}', np.bincount(c["label"]))
    print(c.groupby('label')['Age'].agg(['count', 'mean', 'std']).round(1), '\n')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def age_baseline(c, name):
    X = c[['Age']].values
    y = c['label'].values
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, class_weight=None))
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    pred = cross_val_predict(clf, X, y, cv=cv)
    prob = cross_val_predict(clf, X, y, cv=cv, method='predict_proba')

    acc = accuracy_score(y, pred)
    f1  = f1_score(y, pred, average='macro')
    auc = roc_auc_score(y, prob, multi_class='ovr', average='macro')
    maj = np.bincount(y).max() / len(y)

    print(f'{name}  (n={len(y)})')
    print(f'  majority baseline : {maj:.3f}')
    print(f'  accuracy          : {acc:.3f}')
    print(f'  macro F1          : {f1:.3f}')
    print(f'  macro AUC         : {auc:.3f}\n')
    return acc, f1, auc

age_baseline(A, 'A  buggy cohort, AGE ONLY')
age_baseline(B, 'B  corrected cohort, AGE ONLY')

In [ ]:
import numpy as np

d = np.load('/content/oasis1_slices_3ch.npz')
IMGS, NPZ_LABELS, NPZ_IDS = d['imgs'], d['labels'], d['ids']
pos = {sid: i for i, sid in enumerate(NPZ_IDS)}

B = B[B['ID'].isin(pos)].copy()
B['img_idx'] = B['ID'].map(pos)

print(len(B), 'subjects aligned')
print('labels match:', (NPZ_LABELS[B['img_idx'].values] == B['label'].values).all())
print(IMGS.shape)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

LABELS = B['label'].values
IDX    = B['img_idx'].values
AGE    = B['Age'].values.astype(np.float32)
NCLS   = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, len(LABELS))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.ToPILImage(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return self.tf(IMGS[IDX[r]]), torch.tensor(int(LABELS[r]))

In [ ]:
def run_fold(tr, te, seed=42):
    torch.manual_seed(seed)
    trl = DataLoader(DS(tr, train_tf), batch_size=16, shuffle=True, drop_last=True)
    tel = DataLoader(DS(te, eval_tf), batch_size=16)

    m = models.resnet18(weights='IMAGENET1K_V1')
    m.fc = nn.Linear(m.fc.in_features, NCLS); m = m.to(device)

    cnt = np.bincount(LABELS[tr], minlength=NCLS)
    w = torch.tensor(np.sqrt(len(tr) / (NCLS * cnt)), dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

    def go(ep, o, s):
        for _ in range(ep):
            m.train()
            for x, y in trl:
                x, y = x.to(device), y.to(device)
                o.zero_grad(); crit(m(x), y).backward(); o.step()
            s.step()

    for p in m.parameters(): p.requires_grad = False
    for p in m.fc.parameters(): p.requires_grad = True
    o = torch.optim.AdamW(m.fc.parameters(), lr=1e-3, weight_decay=1e-4)
    go(25, o, torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=25))

    for p in m.parameters(): p.requires_grad = True
    o = torch.optim.AdamW([
        {'params': m.layer1.parameters(), 'lr': 1e-6},
        {'params': m.layer2.parameters(), 'lr': 1e-6},
        {'params': m.layer3.parameters(), 'lr': 1e-5},
        {'params': m.layer4.parameters(), 'lr': 1e-5},
        {'params': m.fc.parameters(),     'lr': 1e-4}], weight_decay=1e-4)
    go(15, o, torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=15))

    m.eval(); P = []
    with torch.no_grad():
        for x, _ in tel:
            P.append(torch.softmax(m(x.to(device)), 1).cpu().numpy())
    return np.concatenate(P)

In [ ]:
from sklearn.model_selection import StratifiedKFold

rows = np.arange(len(LABELS))
OOF  = np.zeros((len(rows), NCLS))

for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(rows, LABELS), 1):
    OOF[te] = run_fold(tr, te)
    print('fold', k, 'done')

np.save('/content/oof_probs.npy', OOF)
print(OOF.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

y  = LABELS
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def lr_oof(X):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_predict(clf, X, y, cv=cv, method='predict_proba')

P_age  = lr_oof(AGE.reshape(-1, 1))
P_mri  = OOF
P_both = lr_oof(np.column_stack([AGE, np.log(OOF + 1e-6)]))

def score(P):
    return (accuracy_score(y, P.argmax(1)),
            f1_score(y, P.argmax(1), average='macro'),
            roc_auc_score(y, P, multi_class='ovr', average='macro'))

print(f'{"model":<12}{"acc":>8}{"macroF1":>10}{"AUC":>8}')
for name, P in [('age only', P_age), ('MRI only', P_mri), ('age + MRI', P_both)]:
    a, f, u = score(P)
    print(f'{name:<12}{a:>8.3f}{f:>10.3f}{u:>8.3f}')

In [ ]:
rng = np.random.default_rng(0)
B_ITERS = 2000
models_ = {'age': P_age, 'mri': P_mri, 'both': P_both}
aucs = {k: [] for k in models_}
diffs = {'mri - age': [], 'both - mri': [], 'both - age': []}

n = len(y)
for _ in range(B_ITERS):
    s = rng.integers(0, n, n)
    if len(np.unique(y[s])) < 3:
        continue
    a = {k: roc_auc_score(y[s], P[s], multi_class='ovr', average='macro') for k, P in models_.items()}
    for k in a: aucs[k].append(a[k])
    diffs['mri - age'].append(a['mri'] - a['age'])
    diffs['both - mri'].append(a['both'] - a['mri'])
    diffs['both - age'].append(a['both'] - a['age'])

def ci(v): return np.percentile(v, [2.5, 97.5])

print('AUC with 95% CI')
for k in aucs:
    lo, hi = ci(aucs[k]); print(f'  {k:<6} {np.mean(aucs[k]):.3f}  [{lo:.3f}, {hi:.3f}]')

print('\nPaired differences (95% CI — excludes 0 means real)')
for k in diffs:
    lo, hi = ci(diffs[k]); m = np.mean(diffs[k])
    verdict = 'REAL' if (lo > 0 or hi < 0) else 'not distinguishable'
    print(f'  {k:<12} {m:+.3f}  [{lo:+.3f}, {hi:+.3f}]   {verdict}')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

dA = np.load('/content/oasis1_slices_3ch_all436.npz')
pos_A = {sid: i for i, sid in enumerate(dA['ids'])}

A = A[A['ID'].isin(pos_A)].copy()
A['img_idx'] = A['ID'].map(pos_A)
A['subject'] = A['ID'].str.extract(r'(OAS1_\d{4})')[0]

print(len(A), 'sessions,', A['subject'].nunique(), 'unique people')
print('labels match:', (dA['labels'][A['img_idx'].values] == A['label'].values).all())

cvA = list(StratifiedGroupKFold(5, shuffle=True, random_state=42)
           .split(A, A['label'], groups=A['subject']))

for k, (tr, te) in enumerate(cvA, 1):
    overlap = set(A['subject'].iloc[tr]) & set(A['subject'].iloc[te])
    print(f'fold {k}: test n={len(te)}  classes={np.bincount(A["label"].values[te], minlength=3)}  shared people={len(overlap)}')

In [ ]:
IMGS_B, LABELS_B, IDX_B, AGE_B = IMGS, LABELS, IDX, AGE

IMGS   = dA['imgs']
LABELS = A['label'].values
IDX    = A['img_idx'].values
AGE_A  = A['Age'].values.astype(np.float32)

OOF_A = np.zeros((len(LABELS), 3))
for k, (tr, te) in enumerate(cvA, 1):
    OOF_A[te] = run_fold(tr, te)
    print('fold', k, 'done')

np.save('/content/oof_probs_cohortA.npy', OOF_A)
print(OOF_A.shape)

In [ ]:
def oof_lr(X, y, folds):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_predict(clf, X, y, cv=folds, method='predict_proba')

yA, yB = A['label'].values, LABELS_B
P_age_A = oof_lr(AGE_A.reshape(-1, 1), yA, cvA)
groups_A = A['subject'].values

def boot_gap(y, P1, P2, groups=None, iters=2000, seed=0):
    rng = np.random.default_rng(seed)
    units = np.unique(groups) if groups is not None else np.arange(len(y))
    rows_of = {u: np.where(groups == u)[0] for u in units} if groups is not None else None
    a1, a2, gap = [], [], []
    for _ in range(iters):
        pick = rng.choice(units, len(units), replace=True)
        s = np.concatenate([rows_of[u] for u in pick]) if groups is not None else pick
        if len(np.unique(y[s])) < 3: continue
        u1 = roc_auc_score(y[s], P1[s], multi_class='ovr', average='macro')
        u2 = roc_auc_score(y[s], P2[s], multi_class='ovr', average='macro')
        a1.append(u1); a2.append(u2); gap.append(u2 - u1)
    c = lambda v: np.percentile(v, [2.5, 97.5])
    return c(a1), c(a2), np.mean(gap), c(gap)

print(f'{"cohort":<22}{"age AUC":>16}{"MRI AUC":>16}{"MRI − age":>26}')
for name, y, Pa, Pm, g in [('A  buggy  (n=436)', yA, P_age_A, OOF_A, groups_A),
                           ('B  fixed  (n=235)', yB, P_age, OOF, None)]:
    ua = roc_auc_score(y, Pa, multi_class='ovr', average='macro')
    um = roc_auc_score(y, Pm, multi_class='ovr', average='macro')
    ca, cm, gm, cg = boot_gap(y, Pa, Pm, g)
    print(f'{name:<22}{ua:>6.3f} [{ca[0]:.2f},{ca[1]:.2f}]'
          f'{um:>6.3f} [{cm[0]:.2f},{cm[1]:.2f}]'
          f'{gm:>+8.3f} [{cg[0]:+.3f},{cg[1]:+.3f}]')

In [ ]:
mask = A['CDR'].notna().values
yE   = yA[mask]

for name, P in [('age only', P_age_A), ('MRI (trained on A)', OOF_A)]:
    print(f'{name:<20} all 436: {roc_auc_score(yA, P, multi_class="ovr", average="macro"):.3f}'
          f'   elderly 235 only: {roc_auc_score(yE, P[mask], multi_class="ovr", average="macro"):.3f}')

In [ ]:
from itertools import product

def apply(P, b): return (np.log(P + 1e-9) + b).argmax(1)

grid = [np.array([0, b1, b2]) for b1, b2 in product(np.arange(-1.5, 1.6, 0.25), repeat=2)]
pred = np.zeros(len(y), int)

for tr, te in StratifiedKFold(5, shuffle=True, random_state=7).split(OOF, y):
    best = max(grid, key=lambda b: accuracy_score(y[tr], apply(OOF[tr], b)))
    pred[te] = apply(OOF[te], best)

print('tuned accuracy :', round(accuracy_score(y, pred), 3))
print('AD recall      :', round((pred[y == 2] == 2).mean(), 3))
print('AUC unchanged  :', round(roc_auc_score(y, OOF, multi_class='ovr', average='macro'), 3))